# AdiOS Regulatory AI — Pipeline Demo

This notebook demonstrates the end-to-end pipeline for a synthetic SAE document.


In [ ]:
import httpx, json
BASE = 'http://localhost:8000/api/v1'

# Step 1: Anonymise the raw SAE document
raw_sae = open('../data/sample/sample_clinical_document.txt').read()
r = httpx.post(f'{BASE}/anonymise', json={'text': raw_sae, 'mode': 'anonymise', 'document_type': 'sae_report'})
anon = r.json()
print(f"Entities detected: {len(anon['entities_detected'])}")
print(f"Compliance flags: {anon['compliance_flags']}")

In [ ]:
# Step 2: Classify the SAE
r2 = httpx.post(f'{BASE}/classify-sae', json={'case_narration': anon['anonymised_text'], 'check_duplicate': False})
cls = r2.json()
print(f"Severity: {cls['severity']}  Priority: {cls['priority']}  Confidence: {cls['confidence']}")
print(f"Expedited reporting required: {cls['expedited_reporting_required']}")

In [ ]:
# Step 3: Summarise the SAE narration
r3 = httpx.post(f'{BASE}/summarise', json={'text': anon['anonymised_text'], 'source_type': 'sae_narration'})
summ = r3.json()
print('Summary:', summ['summary'])
print('Action items:', summ['action_items'])

In [ ]:
# Step 4: Completeness check on the SAE report
import json
sae_doc = json.load(open('../data/sample/sample_sae_report.json'))
r4 = httpx.post(f'{BASE}/assess-completeness', json={'document': sae_doc, 'schema_type': 'sae_report'})
comp = r4.json()
print(f"Complete: {comp['is_complete']}  Score: {comp['completeness_score']}")
print(f"Recommendation: {comp['review_recommendation']}")